# Config C Multi-Seed Evaluation
Evaluates fine-tuned CLIP + BLIP-2 ITM reranking (Config C) across 3 seeds.
- Seeds: 42, 104, 541
- Alpha values: 0.7, 0.5
- Metrics: Recall@K, NDCG@K, mAP@K for K in {5, 10, 15}


In [1]:
!pip install -q omegaconf timm fairscale iopath decord webdataset pycocotools pycocoevalcap
!pip install -q --no-dependencies salesforce-lavis
!pip install -q transformers==4.38.2 open_clip_torch pinecone pandas Pillow tqdm scikit-learn accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 15.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 32.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 82.5 MB/s eta 

In [2]:
import os
import math
import torch
import numpy as np
import open_clip
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from pinecone import Pinecone
from lavis.models import load_model_and_preprocess

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

/usr/local/lib/python3.12/dist-packages/timm/models/hub.py:4: FutureWarning: Importing from timm.models.hub is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__

Device: cuda


In [3]:
# ==============================================================================
# CONFIGURATIONS
# ==============================================================================
PINECONE_API_KEY = "pcsk_REDACTED"  # Replace with your key
INDEX_NAME       = "vr-clothing-gallery"

DATASET_ROOT = "/kaggle/input/datasets/vinay1706/deepfashion-cropped"
QUERY_CSV    = f"{DATASET_ROOT}/query.csv"
GALLERY_CSV  = f"{DATASET_ROOT}/gallery.csv"
CAPTIONS_CSV = f"{DATASET_ROOT}/blip2_captions_gallery.csv"
IMAGE_ROOT   = DATASET_ROOT

TOP_K_RETRIEVAL = 15
K_VALUES        = [5, 10, 15]

SEEDS           = [104, 541]
FINETUNED_ALPHAS = [0.7, 0.5]


In [4]:
# ==============================================================================
# LOAD DATA
# ==============================================================================
query_df    = pd.read_csv(QUERY_CSV)
gallery_df  = pd.read_csv(GALLERY_CSV)
captions_df = pd.read_csv(CAPTIONS_CSV)

caption_map     = dict(zip(captions_df['image_name'], captions_df['blip2_caption']))
relevance_count = gallery_df.groupby('item_id').size().to_dict()

# One query per unique item ID — avoids inflating metrics via duplicate queries
eval_df = query_df.drop_duplicates(subset='item_id').reset_index(drop=True)

print(f"Total query images:      {len(query_df)}")
print(f"Unique item IDs:         {len(eval_df)}")
print(f"Gallery images:          {len(gallery_df)}")
print(f"Captions loaded:         {len(caption_map)}")


Total query images:      14218
Unique item IDs:         3985
Gallery images:          12612
Captions loaded:         12612


In [5]:
# ==============================================================================
# LOAD MODELS
# ==============================================================================
print("Loading LAVIS BLIP-2 ITM model...")
blip_model, vis_processors, txt_processors = load_model_and_preprocess(
    name="blip2_image_text_matching",
    model_type="pretrain",
    is_eval=True,
    device=device
)

# Fix missing padding in batched LAVIS inference
class PatchedTokenizer:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
    def __call__(self, *args, **kwargs):
        kwargs["padding"] = True
        return self.tokenizer(*args, **kwargs)
    def __getattr__(self, name):
        return getattr(self.tokenizer, name)

blip_model.tokenizer = PatchedTokenizer(blip_model.tokenizer)
print("BLIP-2 ITM model loaded.")

def load_clip(checkpoint_path):
    model, _, preprocess = open_clip.create_model_and_transforms('ViT-L-14', pretrained='openai')
    ckpt = torch.load(checkpoint_path, map_location=device)
    state_dict = ckpt.get("model_state", ckpt)
    state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict, strict=False)
    print(f"  Loaded checkpoint: {checkpoint_path}")
    return model.to(device).eval(), preprocess

print("Connecting to Pinecone...")
pc    = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(INDEX_NAME)
print("Connected.")


Loading LAVIS BLIP-2 ITM model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

100%|██████████| 1.89G/1.89G [00:08<00:00, 228MB/s]


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

100%|██████████| 712M/712M [00:05<00:00, 134MB/s]


BLIP-2 ITM model loaded.
Connecting to Pinecone...
Connected.


In [6]:
# ==============================================================================
# BLIP-2 ITM SCORING
# Accepts a precomputed image tensor (computed once per query, outside this fn).
# Batches all candidate captions in a single forward pass.
# Returns P(match) in [0,1] for each candidate.
# ==============================================================================
@torch.no_grad()
def compute_itm_scores_batched(img_tensor, candidate_captions):
    # img_tensor: (1, C, H, W) already on device — precomputed outside
    img_batch  = img_tensor.repeat(len(candidate_captions), 1, 1, 1)
    txt_batch  = [txt_processors["eval"](c) for c in candidate_captions]
    itm_output = blip_model({"image": img_batch, "text_input": txt_batch}, match_head="itm")
    itm_scores = torch.nn.functional.softmax(itm_output, dim=1)[:, 1].cpu().tolist()
    return itm_scores


In [7]:
# ==============================================================================
# METRICS
# Definitions follow https://www.evidentlyai.com/ranking-metrics/ndcg-metric
#                 and https://www.evidentlyai.com/ranking-metrics/mean-average-precision-map
#
# Recall@K  : relevant items retrieved in top-K / total relevant items in gallery
# NDCG@K    : DCG normalised by ideal DCG; accumulates gain for every relevant hit
# mAP@K     : precision at every relevant rank position, averaged over total relevant items
# ==============================================================================
def calculate_metrics(ranked_item_ids, ground_truth_id, k_values=[5, 10, 15]):
    metrics      = {}
    total_relevant = relevance_count.get(ground_truth_id, 1)

    for k in k_values:
        top_k_items = ranked_item_ids[:k]

        # Recall@K
        relevant_in_top_k = sum(1 for item in top_k_items if item == ground_truth_id)
        metrics[f'Recall@{k}'] = relevant_in_top_k / total_relevant

        # NDCG@K
        dcg = sum(
            1.0 / math.log2(i + 2)
            for i, item in enumerate(top_k_items)
            if item == ground_truth_id
        )
        n_ideal = min(total_relevant, k)
        idcg    = sum(1.0 / math.log2(i + 2) for i in range(n_ideal))
        metrics[f'NDCG@{k}'] = (dcg / idcg) if idcg > 0 else 0.0

        # mAP@K
        ap = 0.0
        relevant_so_far = 0
        for i, item in enumerate(top_k_items):
            if item == ground_truth_id:
                relevant_so_far += 1
                ap += relevant_so_far / (i + 1)
        metrics[f'mAP@{k}'] = ap / total_relevant

    return metrics


In [8]:
# ==============================================================================
# MAIN EVALUATION LOOP — Config C, all seeds
# ==============================================================================
all_results = []

for seed in SEEDS:
    checkpoint = f"{DATASET_ROOT}/clip_best_seed{seed}.pt"
    print(f"\n{'#'*60}\n# SEED {seed}\n{'#'*60}")

    if not os.path.exists(checkpoint):
        print(f"  WARNING: {checkpoint} not found — skipping seed {seed}")
        continue

    clip_model, clip_preprocess = load_clip(checkpoint)

    for alpha in FINETUNED_ALPHAS:
        namespace   = f"finetuned-alpha-{alpha}-seed{seed}"
        config_name = f"Config C (FT, alpha={alpha}, seed={seed})"
        print(f"\n{'='*60}\nEvaluating {config_name}\n{'='*60}")

        config_metrics = {
            f'{m}@{k}': []
            for m in ['Recall', 'NDCG', 'mAP']
            for k in K_VALUES
        }

        for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
            query_image_path = os.path.join(IMAGE_ROOT, row['image_name'])
            gt_item_id       = row['item_id']

            try:
                raw_image = Image.open(query_image_path).convert('RGB')
            except Exception as e:
                print(f"  Skipping {query_image_path}: {e}")
                continue

            # Preprocess once — reused for both CLIP and ITM
            processed_image = clip_preprocess(raw_image).unsqueeze(0).to(device)
            img_tensor      = vis_processors["eval"](raw_image).unsqueeze(0).to(device)

            # CLIP embedding
            with torch.no_grad():
                with torch.amp.autocast('cuda'):
                    query_embedding = clip_model.encode_image(processed_image)
                    query_embedding = torch.nn.functional.normalize(
                        query_embedding, dim=-1
                    ).cpu().numpy().tolist()[0]

            # Pinecone retrieval
            retrieval_response = index.query(
                vector=query_embedding,
                top_k=TOP_K_RETRIEVAL,
                namespace=namespace,
                include_metadata=True
            )
            candidates = retrieval_response['matches']

            # BLIP-2 ITM reranking
            cand_images   = [c['metadata']['image_name'] for c in candidates]
            cand_captions = [caption_map.get(img, "") for img in cand_images]
            itm_scores    = compute_itm_scores_batched(img_tensor, cand_captions)

            scored     = sorted(zip(candidates, itm_scores), key=lambda x: x[1], reverse=True)
            candidates = [s[0] for s in scored]

            ranked_item_ids = [c['metadata']['item_id'] for c in candidates]
            q_metrics       = calculate_metrics(ranked_item_ids, gt_item_id, K_VALUES)

            for key, val in q_metrics.items():
                config_metrics[key].append(val)

        # Aggregate
        print(f"\nResults for {config_name}:")
        summary_row = {"Config": config_name, "Seed": seed, "Alpha": alpha}
        for metric, values in config_metrics.items():
            mean_val = float(np.mean(values))
            summary_row[metric] = mean_val
            print(f"  {metric}: {mean_val:.4f}")
        all_results.append(summary_row)

    del clip_model
    torch.cuda.empty_cache()

# ==============================================================================
# SUMMARY TABLE
# ==============================================================================
results_df = pd.DataFrame(all_results)
col_order  = ["Config", "Seed", "Alpha"] + [
    f'{m}@{k}' for k in K_VALUES for m in ['Recall', 'NDCG', 'mAP']
]
results_df = results_df[col_order]

display(results_df)
results_df.to_csv("/kaggle/working/config_c_multiseed_results.csv", index=False)
print("\nSaved to /kaggle/working/config_c_multiseed_results.csv")

# ==============================================================================
# MEAN +/- STD ACROSS SEEDS (per alpha)
# ==============================================================================
print("\nMean ± Std across seeds:")
metric_cols = [f'{m}@{k}' for k in K_VALUES for m in ['Recall', 'NDCG', 'mAP']]
for alpha in FINETUNED_ALPHAS:
    subset = results_df[results_df['Alpha'] == alpha]
    print(f"\n  Alpha = {alpha}")
    for col in metric_cols:
        mean = subset[col].mean()
        std  = subset[col].std()
        print(f"    {col}: {mean:.4f} ± {std:.4f}")



############################################################
# SEED 104
############################################################


open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


  Loaded checkpoint: /kaggle/input/datasets/vinay1706/deepfashion-cropped/clip_best_seed104.pt

Evaluating Config C (FT, alpha=0.7, seed=104)


  0%|          | 0/3985 [00:00<?, ?it/s]

...


Results for Config C (FT, alpha=0.7, seed=541):
  Recall@5: 0.6582
  Recall@10: 0.8425
  Recall@15: 0.9421
  NDCG@5: 0.6086
  NDCG@10: 0.6800
  NDCG@15: 0.7194
  mAP@5: 0.4971
  mAP@10: 0.5535
  mAP@15: 0.5811

Evaluating Config C (FT, alpha=0.5, seed=541)


  0%|          | 0/3985 [00:00<?, ?it/s]


Results for Config C (FT, alpha=0.5, seed=541):
  Recall@5: 0.6265
  Recall@10: 0.8237
  Recall@15: 0.9323
  NDCG@5: 0.5746
  NDCG@10: 0.6516
  NDCG@15: 0.6941
  mAP@5: 0.4646
  mAP@10: 0.5218
  mAP@15: 0.5502


,Config,Seed,Alpha,Recall@5,NDCG@5,mAP@5,Recall@10,NDCG@10,mAP@10,Recall@15,NDCG@15,mAP@15
0,"Config C (FT, alpha=0.7, seed=104)",104,0.7,0.654664,0.611259,0.501843,0.837571,0.682560,0.558083,0.941911,0.723836,0.586871
1,"Config C (FT, alpha=0.5, seed=104)",104,0.5,0.629944,0.578896,0.469288,0.815371,0.651128,0.523615,0.931836,0.696587,0.553654
2,"Config C (FT, alpha=0.7, seed=541)",541,0.7,0.658175,0.608554,0.497110,0.842497,0.680033,0.553455,0.942144,0.719354,0.581086
3,"Config C (FT, alpha=0.5, seed=541)",541,0.5,0.626507,0.574633,0.464570,0.823726,0.651637,0.521772,0.932329,0.694139,0.550226



Saved to /kaggle/working/config_c_multiseed_results.csv

Mean ± Std across seeds:

  Alpha = 0.7
    Recall@5: 0.6564 ± 0.0025
    NDCG@5: 0.6099 ± 0.0019
    mAP@5: 0.4995 ± 0.0033
    Recall@10: 0.8400 ± 0.0035
    NDCG@10: 0.6813 ± 0.0018
    mAP@10: 0.5558 ± 0.0033
    Recall@15: 0.9420 ± 0.0002
    NDCG@15: 0.7216 ± 0.0032
    mAP@15: 0.5840 ± 0.0041

  Alpha = 0.5
    Recall@5: 0.6282 ± 0.0024
    NDCG@5: 0.5768 ± 0.0030
    mAP@5: 0.4669 ± 0.0033
    Recall@10: 0.8195 ± 0.0059
    NDCG@10: 0.6514 ± 0.0004
    mAP@10: 0.5227 ± 0.0013
    Recall@15: 0.9321 ± 0.0003
    NDCG@15: 0.6954 ± 0.0017
    mAP@15: 0.5519 ± 0.0024
